# 26. Plasticity: the baseline that is missing

Experiment A from `ops-docs/guides/plasticity-probes.html`, arms A0 and A1.

### The question

Plasticity means the ability to learn something new. Loss of plasticity means that ability
gets worse as more tasks are learned. This is a different thing from forgetting, which is
about old tasks getting worse.

So: after eight tasks, does this generator learn a brand new task worse than a fresh
generator would?

### What already exists

Most of this experiment has already been run, under different names. The study's own job
list contains exactly the arms this experiment needs:

| Arm | What it is | Study job | Status |
| --- | --- | --- | --- |
| **A1** | After 8 tasks, then learn task 15 | `controls/order_01/seed_N/learn_15` | Done, 3 seeds |
| **A2** | After 8 tasks, unlearn one, then learn task 15 | `branches/.../forget_T/learn_15` | Done, 6 runs |
| **A0** | Fresh generator, learn task 15, nothing else | none | **Missing** |

A1 and A2 are read from R2 in section 4. Nothing is re-run.

A0 is the one number nobody has. Without it, A1 has nothing to be compared against, so
"the model reached 43% on a new task" cannot be called good or bad. Section 6 runs it.

### What this notebook does

1. Reads the finished A1 and A2 results from R2 and prints them
2. Trains two fresh generators on task 15 alone, which is arm A0
3. Subtracts, and applies the decision rule

### Why A0 is run twice

Arm A0 learns task 15 as its *first* task. Appendix B of the paper freezes the chunk
embeddings after the first task, so a first task is allowed to train something that a
ninth task is not. That difference is nothing to do with plasticity, and it would make A0
look better than it should.

So A0 runs in two versions. **A0** is the plain one. **A0f** freezes the chunk embeddings,
matching the condition arm A1 trained under. The gap between them says how much of any
result is this structural difference rather than plasticity.

### Runtime

Two task-15 learning runs per seed, at 5 epochs each. Everything else is reading files.

## 1. Load the fixed code

In [ ]:
from pathlib import Path
import sys, os, json, subprocess, importlib.util
IN_COLAB = "google.colab" in sys.modules
CODE_VERSION = "65e2e7c850fd6f1852a99d9e81780c27f799c926"
if any(name == "uncle" or name.startswith("uncle.") for name in sys.modules):
    raise RuntimeError("Restart the runtime before loading this fixed code revision.")
if IN_COLAB:
    REPO = Path("/content/cark-paired-study")
    if not REPO.exists():
        subprocess.run(["git", "clone", "https://github.com/sumitasthana/CARK.git", str(REPO)], check=True)
    if subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain", "--untracked-files=no"], text=True).strip():
        raise RuntimeError("Checkout has edits. Use a fresh runtime.")
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", CODE_VERSION], check=True)
else:
    REPO = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "uncle/task_partition.json").exists()), None)
    if REPO is None:
        raise FileNotFoundError("Open this notebook from a CARK checkout.")
    actual = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
    implementation = ["uncle", "pyproject.toml", "requirements.txt"]
    dirty = subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain", "--", *implementation], text=True).strip()
    # A later documentation-only commit may contain the same fixed implementation.
    differences = subprocess.check_output(["git", "-C", str(REPO), "diff", CODE_VERSION, actual, "--", *implementation], text=True).strip()
    if dirty or differences:
        raise RuntimeError("Use a clean checkout of CODE_VERSION, or open in Colab. Local training edits are preserved.")
for package in ("boto3", "tqdm", "matplotlib"):
    if importlib.util.find_spec(package) is None:
        if not IN_COLAB:
            raise ImportError("Install the storage and extras dependencies before running this notebook.")
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", package], check=True)
sys.path.insert(0, str(REPO))
from uncle.storage import R2Store
from uncle.paired_study import StudyStore, jobs
from uncle import checkpoint
from uncle.config import Config
from uncle.hypernet import build_target, HyperNetwork
from uncle.trainer import UnCLe
from uncle.streams import build_tasks

import torch
print("Fixed implementation revision:", CODE_VERSION)
print("PyTorch:", torch.__version__, "CUDA build:", torch.version.cuda)

## 2. Settings

Everything here matches the saved study. The only new choice is `SEEDS`, which says how
many fresh generators to train for arm A0.

One seed gives a usable first answer. Three lets the result be stated with an interval,
and three is what the finished arms already have.

In [ ]:
STUDY_ID = "paired_generalization_v2"
BUCKET = "uncle-experiments"

ORDER = "order_01"

# Which seeds to train a fresh generator for. Arms A1 and A2 already exist for
# seeds 0, 1 and 2, so A0 must cover the same seeds to be compared with them.
# Start with [0] to see one answer quickly, then extend to [0, 1, 2].
SEEDS = [0]

# Freeze the chunk embeddings in the second A0 version. See the header: a first
# task is allowed to train them and a ninth task is not, so the plain A0 enjoys
# an advantage that has nothing to do with plasticity.
RUN_FROZEN_CHUNKS = True

# Below this, a model has stopped responding to its input and its accuracy means
# nothing. The conclusion report measured about 1e-7 for a collapsed model and
# about 11 to 14 for a working one, so this threshold is far from both.
SENSITIVITY_FLOOR = 1e-3

# Two identical runs in this study differed by about 3 accuracy points, because
# the graphics card does not always add numbers in the same order. Anything
# smaller than this is not a finding.
NOISE_FLOOR_POINTS = 3.0

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATA_ROOT = Path("/content/data/tiny-imagenet-200") if IN_COLAB else REPO / "data/tiny-imagenet-200"
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
print("Fresh generators to train:", len(SEEDS) * (2 if RUN_FROZEN_CHUNKS else 1))

# Matching the study runner: no fast paths that change the arithmetic.
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

## 3. Connect to R2

Supply `R2_ENDPOINT`, `R2_ACCESS_KEY_ID`, and `R2_SECRET_ACCESS_KEY` through Colab Secrets
or environment variables. Their values are never printed or saved.

In [ ]:
def secret(name):
    value = os.environ.get(name)
    if not value and IN_COLAB:
        from google.colab import userdata
        value = userdata.get(name)
    if not value:
        raise RuntimeError("Missing secret: " + name)
    return value

store = R2Store.connect(secret("R2_ENDPOINT"), BUCKET,
    secret("R2_ACCESS_KEY_ID"), secret("R2_SECRET_ACCESS_KEY"))
store.progress = True
CACHE = REPO / "outputs/paired_study" / STUDY_ID
CACHE.mkdir(parents=True, exist_ok=True)
print("Connected to R2 bucket:", BUCKET)

## 4. Read the arms that already exist

Nothing is trained in this section. It downloads the finished reports and reads one number
out of each: the accuracy the model reached on task 15.

- A **control** job learned the eight tasks of order 01 and then learned task 15. That is
  arm A1.
- A **branch** job did the same, but unlearned one task before learning task 15. That is
  arm A2.

In [ ]:
PREFIX = "uncle/paired_generalization/" + STUDY_ID
path = store.download(PREFIX + "/study.json", CACHE / "cache")
plan = json.loads(path.read_text())

if plan["code_version"] != CODE_VERSION:
    raise ValueError("This study requires a different fixed code revision.")
from uncle.learning_diagnostic import file_hash
if file_hash(REPO / "uncle/task_partition.json") != plan["partition_sha256"]:
    raise ValueError("Class partition differs from the saved study.")

storage = StudyStore(store, plan, CACHE, cleanup_completed=False)
ALL_JOBS = jobs(plan)
ORDER_TASKS = tuple(plan["orders"][ORDER])
NEW_TASK = plan["new_tasks"][0]          # task 15, the probe
CLASSES = plan["config"]["classes_per_task"]
EVAL_BATCH = plan["config"]["eval_batch_size"]

def finished(job_id):
    """Return a completed job's report, or None if it was never finished.

    storage.completed_report does the careful part: it returns None for a job
    that is missing or half done, checks the saved file against the size and
    checksum the study recorded, and refuses a report that belongs to a
    different job. A branch whose model file was deleted after verification
    still has its report, which is all this notebook reads.
    """
    job = next((j for j in ALL_JOBS if j["id"] == job_id), None)
    return storage.completed_report(job) if job else None

existing = {}
for seed in plan["seeds"]:
    base = f"{ORDER}/seed_{seed}"
    report = finished(f"controls/{base}/learn_{NEW_TASK}")
    if report:
        existing.setdefault("A1", {})[seed] = report["final_accuracies"][NEW_TASK]
    for target in plan["forget_tasks"]:
        report = finished(f"branches/{base}/forget_{target}/learn_{NEW_TASK}")
        if report:
            existing.setdefault(f"A2 forget {target}", {})[seed] = report["final_accuracies"][NEW_TASK]

print(f"Task {NEW_TASK} accuracy in the arms that are already finished\n")
print(f"{'arm':<16}" + "".join(f"seed {s:<6}" for s in plan["seeds"]) + "  mean")
for arm, scores in existing.items():
    row = "".join(f"{scores.get(s, float('nan')):<11.1f}" for s in plan["seeds"])
    mean = sum(scores.values()) / len(scores)
    print(f"{arm:<16}{row}  {mean:.1f}")
print("\nThese are A1 and A2. A0 does not exist yet, which is why the numbers above")
print("cannot yet be called high or low.")

## 5. Load the images

Only task 15 is needed here. Arm A0 learns nothing else, and the tasks in order 01 are
never touched by this notebook.

In [ ]:
def fetch_tasks(include):
    """Build the task datasets, clearing a broken download if there is one.

    torchvision reuses any file already on disk and no checksum is given here,
    so a half-finished or error-page download is kept and every rerun fails the
    same way. Delete it and try once more before giving up.
    """
    import shutil, zipfile
    archive = Path(DATA_ROOT).parent / "tiny-imagenet-200.zip"
    for attempt in (1, 2):
        try:
            return build_tasks(root=DATA_ROOT,
                               partition_path=REPO / "uncle/task_partition.json",
                               partition_seed=42, include=include, download=True)
        except (zipfile.BadZipFile, EOFError, OSError) as problem:
            if attempt == 2:
                raise
            print("The Tiny ImageNet download is unusable:", problem)
            if archive.exists():
                head = archive.open("rb").read(4)
                print(f"   {archive} is {archive.stat().st_size} bytes, starts with {head!r}")
                if not head.startswith(b"PK"):
                    print("   That is not a zip, so the server sent something else.")
                archive.unlink()
            shutil.rmtree(Path(DATA_ROOT), ignore_errors=True)
            print("   Cleared it. Downloading again.")

tasks = fetch_tasks((NEW_TASK,))
print("Training images for task", NEW_TASK + ":", len(tasks[NEW_TASK]["train"]))
print("Test images for task", NEW_TASK + ":", len(tasks[NEW_TASK]["test"]))

## 6. The measurement

Accuracy alone cannot tell two very different models apart. A model that guesses scores
about 10 percent. A model that has stopped reading the picture and answers the same class
every time also scores about 10 percent.

Input sensitivity separates them. It asks how far any single output score travels across
the whole test set. A working model moves a lot. A model that ignores its input barely
moves at all.

This is the same function notebook 23 uses, so the numbers are comparable with that work.

In [ ]:
from torch.func import functional_call
from torch.utils.data import DataLoader

def inspect(hypernet, target, trainer, task):
    """Accuracy, how many classes get used, and how far the scores travel.

    The classifier returns ten raw scores per image, one per class. The highest
    one is the answer. Across the test set, each score position has a lowest and
    a highest value it ever reached; the gap between them is that position's
    range. Input sensitivity is the largest of those ten ranges.

    Taking the largest is deliberately generous to the model: one output that
    still moves is enough to count it as alive.
    """
    weights = hypernet.weights_for(task)
    buffers = {name: value.clone() for name, value in trainer.task_buffers[task].items()}
    loader = DataLoader(tasks[task]["test"], batch_size=EVAL_BATCH)
    picked, correct, total = [], 0, 0
    lowest = highest = None
    with torch.no_grad():
        for images, labels in loader:
            scores = functional_call(target, (weights, buffers),
                                     (images.to(trainer.device),), strict=True)
            choice = scores.argmax(1)
            picked.append(choice.cpu())
            correct += (choice == labels.to(trainer.device)).sum().item()
            total += labels.numel()
            low, high = scores.min(dim=0).values, scores.max(dim=0).values
            lowest = low if lowest is None else torch.minimum(lowest, low)
            highest = high if highest is None else torch.maximum(highest, high)
    counts = torch.bincount(torch.cat(picked), minlength=CLASSES).tolist()
    return {"accuracy": 100.0 * correct / total,
            "distinct_classes": sum(1 for c in counts if c > 0),
            "class_counts": counts,
            "input_sensitivity": (highest - lowest).max().item()}

## 7. Arm A0: train a fresh generator on task 15 alone

This is the part that has never been run.

**Training starts from scratch.** Nothing is restored and no checkpoint is loaded. A new
generator is built with the study's settings and then taught one task.

The protection term is switched off by having nothing to protect. That term pulls the
generator to keep producing the same weights for tasks already learned, and here there are
none, so `protected` is an empty list. This is not a choice: it is what a zero-task model
means.

**Two versions run per seed.** The plain one lets the chunk embeddings train, which is
what the code does on a first task. The frozen one does not, matching the condition arm A1
trained under. The frozen version works by registering a throwaway task code first, so
that task 15 is no longer the first task the generator has seen. No file in `uncle/` is
edited, which keeps the pinned code revision honest.

**Nothing is saved to R2.** The results are written to one local JSON file at the end of
section 8.

In [ ]:
import time

def train_fresh(seed, freeze_chunks):
    """Build a generator that has learned nothing, then teach it the probe task.

    seed           picks the starting weights and the task code, so it must match
                   the seed of the A1 run this will be compared against.
    freeze_chunks  True registers a throwaway task code before training, so the
                   probe is not treated as the first task and the chunk
                   embeddings stay fixed.

    Returns the measurement dictionary from inspect(), plus the loss per epoch.
    """
    # Everything from the saved study except the seed, the device, and the
    # request list. One request: learn the probe task, nothing before it.
    config = Config(**{**plan["config"], "seed": seed, "device": DEVICE,
                       "requests": (("learn", NEW_TASK),)})
    target = build_target(config)
    hypernet = HyperNetwork(target, config)
    trainer = UnCLe(hypernet, config, target, tasks, progress=True)

    if freeze_chunks:
        # trainer.learn decides whether chunk embeddings may train by checking
        # whether this is the only task code that exists. Adding a code that is
        # never trained or evaluated makes the answer no. Task codes are drawn
        # from a stream keyed by task name, so this draw cannot change the code
        # task 15 receives.
        hypernet.add_task("__chunk_freeze_placeholder__")

    began = time.perf_counter()
    # protected is empty: there are no earlier tasks for the protection term to
    # hold still. losses is one average loss per epoch.
    losses = trainer.learn(NEW_TASK, [])
    seconds = time.perf_counter() - began

    # Evaluation mode matters. In training mode a BatchNorm layer rescales using
    # the batch in front of it, which can revive a nearly dead signal and hide
    # exactly what is being measured.
    hypernet.eval()
    target.eval()
    result = inspect(hypernet, target, trainer, NEW_TASK)
    result.update(seed=seed, frozen_chunks=freeze_chunks,
                  epoch_losses=losses, seconds=seconds)

    print(f"  seed {seed} | chunks {'frozen' if freeze_chunks else 'trainable'} "
          f"| task {NEW_TASK} {result['accuracy']:.1f}% "
          f"| classes used {result['distinct_classes']}/{CLASSES} "
          f"| sensitivity {result['input_sensitivity']:.3e} "
          f"| {seconds/60:.1f} min")
    return result

a0_runs = []
print(f"Training fresh generators on task {NEW_TASK}. Nothing is restored.\n")
for seed in SEEDS:
    a0_runs.append(train_fresh(seed, freeze_chunks=False))
    if RUN_FROZEN_CHUNKS:
        a0_runs.append(train_fresh(seed, freeze_chunks=True))

## 8. The comparison

Two numbers come out of this.

The **plasticity change** is arm A1 minus arm A0, matched seed by seed. Matching matters:
seed 2 scored much lower than seed 0 in the finished runs, so comparing across seeds would
measure the seed and not the model.

A **negative** plasticity change means the generator learns a new task worse after eight
tasks than it did fresh. That is plasticity loss.

The **chunk effect** is the plain A0 minus the frozen A0. It says how much of the
plasticity change is only the structural difference described in the header, and not
plasticity at all.

In [ ]:
import statistics

def mean(values):
    return sum(values) / len(values) if values else float("nan")

# The frozen version is the fair comparison for A1, because A1 trained with its
# chunk embeddings already fixed. The plain version is kept to measure the gap.
fair = {r["seed"]: r for r in a0_runs if r["frozen_chunks"]} or \
       {r["seed"]: r for r in a0_runs if not r["frozen_chunks"]}
plain = {r["seed"]: r for r in a0_runs if not r["frozen_chunks"]}

print("The gate: a model below the sensitivity floor is not measuring plasticity\n")
dead = [r for r in a0_runs if r["input_sensitivity"] < SENSITIVITY_FLOOR]
if dead:
    for r in dead:
        print(f"  VOID seed {r['seed']}: sensitivity {r['input_sensitivity']:.3e} "
              f"is below {SENSITIVITY_FLOOR}. This run measured a dead model.")
else:
    print("  All A0 runs respond to their input. The accuracies below are readable.\n")

paired = []
print(f"{'seed':<7}{'A0 fair':<11}{'A1':<11}{'A1 - A0':<11}")
for seed in sorted(fair):
    if seed not in existing.get("A1", {}):
        print(f"{seed:<7}{fair[seed]['accuracy']:<11.1f}{'not run':<11}")
        continue
    a0, a1 = fair[seed]["accuracy"], existing["A1"][seed]
    paired.append(a1 - a0)
    print(f"{seed:<7}{a0:<11.1f}{a1:<11.1f}{a1 - a0:<+11.1f}")

if paired:
    change = mean(paired)
    spread = statistics.stdev(paired) if len(paired) > 1 else float("nan")
    print(f"\nPlasticity change (A1 minus A0): {change:+.1f} points", end="")
    print(f", sd {spread:.1f} over {len(paired)} seeds" if len(paired) > 1 else " (one seed)")

    if abs(change) < NOISE_FLOOR_POINTS:
        verdict = (f"NO FINDING. {abs(change):.1f} points is inside the "
                   f"{NOISE_FLOOR_POINTS}-point noise floor.")
    elif change < 0:
        verdict = ("PLASTICITY LOSS, provisional. The generator learns a new task "
                   "worse after eight tasks. Run arm A1z next: the protection term "
                   "could explain this on its own.")
    else:
        verdict = ("NO PLASTICITY LOSS. The generator learns a new task at least as "
                   "well after eight tasks as it did fresh.")
    print("\n" + verdict)

if plain and RUN_FROZEN_CHUNKS:
    gaps = [plain[s]["accuracy"] - fair[s]["accuracy"] for s in sorted(plain) if s in fair]
    print(f"\nChunk effect (trainable minus frozen): {mean(gaps):+.1f} points.")
    print("This is the structural advantage a first task enjoys. If it is larger than")
    print("the plasticity change above, the plasticity change is mostly this.")

## 9. Save the result

One JSON file, written next to the study cache. It holds the A0 runs in full and the
finished arms as they were read, so the comparison can be redone without R2 or a GPU.

In [ ]:
import datetime

stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
out = CACHE / "plasticity" / f"experiment_a_{stamp}.json"
out.parent.mkdir(parents=True, exist_ok=True)

record = {
    "experiment": "A: new-task accuracy, arms A0 and A1",
    "guide": "ops-docs/guides/plasticity-probes.html",
    "code_version": CODE_VERSION,
    "study_id": STUDY_ID,
    "order": ORDER,
    "probe_task": NEW_TASK,
    "config": plan["config"],
    "sensitivity_floor": SENSITIVITY_FLOOR,
    "noise_floor_points": NOISE_FLOOR_POINTS,
    "a0_runs": a0_runs,
    "existing_arms": existing,
    "device": DEVICE,
    "gpu": torch.cuda.get_device_name(0) if DEVICE == "cuda" else None,
}
out.write_text(json.dumps(record, indent=2) + "\n", encoding="utf-8")
print("Saved:", out)
print("\nShare this file, plus the two printed numbers: the plasticity change")
print("and the chunk effect.")

## 10. What comes next

Read the result against the guide's decision rule in section 1.6.

**If the plasticity change is inside the noise floor,** there is no plasticity loss to
report at this probe, and the next useful run is a second probe task. Tasks 2, 4 and 6 are
also unseen in order 01, and one probe task can be unusually easy by accident.

**If the plasticity change is negative and larger than the noise floor,** do not report it
yet. Run arm A1z first. That arm repeats A1 with the protection term switched off
(`beta = 0`), using `run_learning_diagnostic` on the source checkpoint. The protection term
exists to hold the generator still, so it could produce this whole result by working
correctly. Until A1z is run, the two explanations cannot be told apart.

**Either way, arm A2 is already in section 4.** The paper states that unlearning
"restores the hypernetwork's plasticity". A2 minus A1 measures that sentence directly, and
the study's own saved summary already reports this difference per seed. Compare it against
the noise floor before describing it as an effect.